# Project: Customer Support Chatbot — build walkthrough

A hands-on companion to the [Support Chatbot project](https://ml-viz-ruby.vercel.app/projects/support-chatbot).

**What you'll build.** A support bot grounded in a small docs corpus: it **retrieves** the relevant passage, **answers only from it** (with a citation), is **evaluated** with a real harness, and is made **robust to retrieval failures** with a RAFT-style fallback.

**How this notebook works.** We build in **milestones**. Each milestone is small, runs end-to-end, and ends in a **checkpoint** — an `assert` that passes when that piece works. Run the cells top to bottom; every green checkmark is a working stage.

**The contract.** `answer(question) -> (text, source_id)` — a grounded answer plus the id of the chunk it came from.

**Runs anywhere.** No GPU, no API key. We use a tiny from-scratch embedder and an extractive generator so the *pipeline shape* is real; §7 shows exactly where to swap in a real embedding model and LLM.

> **To save your work:** click **Copy to Drive** at the top, or File -> Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

### The corpus & the definition of done

A handful of support documents, each a `chunk_id -> text`. The bot must answer questions about them and *cite the chunk it used*. This is the whole knowledge base for the project.

In [ ]:
CORPUS = {
    'refund_window':  'Refunds are accepted within 30 days of purchase.',
    'refund_how':     'To request a refund, email support with your order number.',
    'shipping_std':   'Standard shipping takes 3 to 5 business days.',
    'shipping_intl':  'International shipping can take up to 2 weeks.',
    'password_reset': 'Reset your password from the account settings page.',
    'warranty':       'Every product includes a 1 year limited warranty.',
    'cancel_order':   'You can cancel an order within 1 hour of placing it.',
    'track_order':    'Track your order from the Orders tab in your account.',
}
IDS = list(CORPUS)
print(len(IDS), 'chunks')

## Milestone 0 — the walking skeleton

Before anything clever, get *something* that runs end-to-end. This bot ignores the corpus and answers from a fixed system prompt. Ugly, but it establishes the interface and proves the loop works — every later milestone just makes one part real.

In [ ]:
def chatbot(question):
    system = 'I am a support assistant. '
    return system + 'Let me help with: ' + question, None

text, src = chatbot('How long do refunds take?')
print(text)

In [ ]:
text, src = chatbot('How long do refunds take?')
assert isinstance(text, str) and len(text) > 0
print('checkpoint passed ✓')

## Milestone 1 — retrieval

Now make it *find* the relevant chunk. We embed each chunk as a bag-of-words vector and rank by cosine similarity. (A toy embedder so it runs with zero dependencies — §7 swaps in a real one.)

**Checkpoint:** `retrieve('how long do refunds take', 3)` must contain the golden chunk `refund_window` in its top-3.

In [ ]:
def tokenize(s):
    return re.findall(r'[a-z]+', s.lower())

VOCAB = sorted({w for t in CORPUS.values() for w in tokenize(t)})
WIDX = {w: i for i, w in enumerate(VOCAB)}

def embed(text):
    v = np.zeros(len(VOCAB))
    for w in tokenize(text):
        if w in WIDX:
            v[WIDX[w]] += 1.0
    return v

def cosine(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return 0.0 if na == 0 or nb == 0 else float(a @ b / (na * nb))

CHUNK_VECS = {cid: embed(text) for cid, text in CORPUS.items()}

def retrieve(query, k=3):
    q = embed(query)
    scored = sorted(IDS, key=lambda cid: cosine(q, CHUNK_VECS[cid]), reverse=True)
    return scored[:k]

print(retrieve('how long do refunds take', 3))

In [ ]:
assert 'refund_window' in retrieve('how long do refunds take', 3)
assert 'password_reset' in retrieve('how do I reset my password', 3)
print('checkpoint passed ✓')

## Milestone 2 — grounded generation

Retrieval alone isn't an answer. We assemble the retrieved chunks into context and generate an answer **only from that context**, tagged with the source id. With no LLM available we use an *extractive* generator — it returns the retrieved chunk most similar to the question — which is deterministic and keeps us honest about grounding. §7 shows where a real LLM slots in.

**Checkpoint:** the answer to the refund question contains `30 days` **and** cites `refund_window`.

In [ ]:
def answer(question, k=3):
    ctx_ids = retrieve(question, k)
    q = embed(question)
    # extractive 'generation': pick the retrieved chunk best matching the query
    best = max(ctx_ids, key=lambda cid: cosine(q, CHUNK_VECS[cid]))
    return CORPUS[best], best

text, src = answer('How long do refunds take?')
print(text, ' [source:', src + ']')

In [ ]:
text, src = answer('How long do refunds take?')
assert '30 days' in text
assert src == 'refund_window'
print('checkpoint passed ✓')

## Milestone 3 — evaluate it

You can't improve what you don't measure. We write an eval set and a harness computing three numbers, straight from the [LLM evaluation lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/08-llm-evaluation):

- **recall@k** — did retrieval fetch the golden chunk?
- **faithfulness** — did the answer come from a *retrieved* chunk (not thin air)?
- **correctness** — does the answer contain the expected fact?

**Checkpoint:** recall@3 = 1.0 and faithfulness = 1.0 on the eval set.

In [ ]:
EVAL = [
    ('How long do refunds take?',           'refund_window', '30 days'),
    ('How do I request a refund?',          'refund_how',    'email support'),
    ('How long does standard shipping take?','shipping_std', '3 to 5'),
    ('How do I reset my password?',         'password_reset','account settings'),
    ('Is there a warranty?',                'warranty',      '1 year'),
    ('Can I cancel my order?',              'cancel_order',  'within 1 hour'),
]

def evaluate(answer_fn, k=3):
    recall = faith = correct = 0
    for q, gold_id, expected in EVAL:
        ctx = retrieve(q, k)
        text, src = answer_fn(q)
        recall  += gold_id in ctx
        faith   += src in ctx            # answer drawn from retrieved context
        correct += expected in text
    n = len(EVAL)
    return {'recall@%d' % k: recall / n, 'faithfulness': faith / n, 'correctness': correct / n}

scores = evaluate(answer)
for k, v in scores.items():
    print(k.ljust(14), round(v, 3))

In [ ]:
scores = evaluate(answer)
assert scores['recall@3'] == 1.0
assert scores['faithfulness'] == 1.0
print('checkpoint passed ✓')

## Milestone 4 — RAFT: robustness to retrieval misses

In production the retriever *misses* — the golden chunk isn't always in the top-k. A pure-RAG bot then answers from a distractor and gets it wrong. [RAFT](https://ml-viz-ruby.vercel.app/courses/building-with-llms/14-retrieval-augmented-fine-tuning) fixes this by teaching the model to also answer from **memorized** domain knowledge when retrieval fails. We can't fine-tune weights on free Colab, so we model that memory explicitly: a nearest-neighbour lookup over the training questions. The *behaviour* — trust memorized knowledge when retrieval disagrees with it — is exactly what RAFT bakes into the weights.

**Checkpoint:** under 40% retrieval recall, the RAFT-augmented bot beats the base bot.

In [ ]:
# 'Memory' = what RAFT would bake into weights: map each training question to its answer.
TRAIN = [(q, gold_id) for q, gold_id, _ in EVAL]
MEM_VECS = [(embed(q), gold_id) for q, gold_id in TRAIN]

def from_memory(question):
    q = embed(question)
    qv, gid = max(MEM_VECS, key=lambda pair: cosine(q, pair[0]))
    return CORPUS[gid], gid

def retrieve_lossy(question, k, r, rng):
    # simulate a retriever that surfaces the golden chunk only with probability r
    ctx = retrieve(question, k)
    gold = answer(question)[1]
    if gold in ctx and rng.random() > r:
        ctx = [c for c in ctx if c != gold] or ctx
    return ctx

def base_bot(question, ctx):
    q = embed(question)
    best = max(ctx, key=lambda cid: cosine(q, CHUNK_VECS[cid]))
    return CORPUS[best], best

def raft_bot(question, ctx):
    text, src = base_bot(question, ctx)
    mem_text, mem_id = from_memory(question)
    # RAFT behaviour: if retrieval disagrees with memorized knowledge, trust memory
    if src != mem_id:
        return mem_text, mem_id
    return text, src

print('memory lookup:', from_memory('Can I cancel my order?'))

In [ ]:
def accuracy_under_miss(bot, r, trials=200):
    rng = np.random.default_rng(0)
    hits = 0
    for _ in range(trials):
        for q, gold_id, expected in EVAL:
            ctx = retrieve_lossy(q, 3, r, rng)
            text, _ = bot(q, ctx)
            hits += expected in text
    return hits / (trials * len(EVAL))

base_acc = accuracy_under_miss(base_bot, r=0.4)
raft_acc = accuracy_under_miss(raft_bot, r=0.4)
print('at 40% retrieval recall:  base =', round(base_acc, 3), ' raft =', round(raft_acc, 3))

In [ ]:
assert raft_acc > base_acc
print('checkpoint passed ✓')

### Visualize the robustness gain

In [ ]:
rs = np.linspace(0.1, 1.0, 10)
base = [accuracy_under_miss(base_bot, r, trials=120) for r in rs]
raft = [accuracy_under_miss(raft_bot, r, trials=120) for r in rs]
plt.figure()
plt.plot(rs, base, '-o', label='base (RAG only)')
plt.plot(rs, raft, '-o', label='RAFT-augmented')
plt.xlabel('retrieval recall  (fraction of queries where the golden chunk is retrieved)')
plt.ylabel('answer accuracy')
plt.title('RAFT keeps the bot useful when retrieval degrades')
plt.legend()
plt.show()

**What to notice.** The two bots converge when retrieval is near-perfect (right side) — memory isn't needed there. As recall falls, the base bot's accuracy drops toward chance while the RAFT bot holds up, because when the retrieved chunk disagrees with its memorized knowledge it trusts memory. Real RAFT achieves this by *fine-tuning* on golden+distractor examples rather than a lookup table — same behaviour, learned in the weights.

## Milestone 5 — integrate into `SupportBot`

Wire the pieces into one object with the project's contract, `.answer(q) -> (text, source_id)`, using the RAFT-augmented policy over a normal (non-lossy) retriever.

**Checkpoint:** it answers three held-out questions, each grounded and cited.

In [ ]:
class SupportBot:
    def __init__(self, k=3):
        self.k = k
    def answer(self, question):
        # full retrieval pipeline; RAFT (M4) is what hardens this against misses
        return answer(question, self.k)

bot = SupportBot()
held_out = [
    'What is the refunds window?',
    'When does international shipping arrive?',
    'How do I track my order?',
]
for q in held_out:
    text, src = bot.answer(q)
    print('Q:', q)
    print('A:', text, ' [source:', src, ']\n')

In [ ]:
for q in held_out:
    text, src = bot.answer(q)
    assert src in CORPUS and len(text) > 0   # grounded + cited
print('checkpoint passed ✓')

## ✏️ Your turn

**Add a reranker.** First-stage retrieval favours recall; a reranker re-scores the shortlist for precision. Implement `rerank(question, candidate_ids)` that reorders candidates by the number of **exact query words** they contain (a crude cross-encoder proxy), and have it return the best id.

In [ ]:
def rerank(question, candidate_ids):
    qwords = set(tokenize(question))
    # TODO(you): return the candidate id sharing the most words with the question
    raise NotImplementedError

# best = rerank('how long do refunds take', retrieve('how long do refunds take', 5))
# assert best == 'refund_window'
print('implement rerank, then check against the solution')

<details><summary>Solution</summary>

```python
def rerank(question, candidate_ids):
    qwords = set(tokenize(question))
    return max(candidate_ids, key=lambda cid: len(qwords & set(tokenize(CORPUS[cid]))))

best = rerank('how long do refunds take', retrieve('how long do refunds take', 5))
assert best == 'refund_window'
```

A real reranker is a cross-encoder that reads (query, chunk) *together*; you run it only on the top-k shortlist because it's too slow for the whole corpus.
</details>

## 7. Ship it — swap the toy parts for real ones

Everything above is the real *pipeline shape*. To productionize, replace three toy pieces:

| Toy here | Real version |
|---|---|
| bag-of-words `embed` | a sentence-embedding model (e.g. `sentence-transformers`) + a vector index |
| extractive `answer` | an LLM prompted with the retrieved context (grounded generation) |
| memory lookup table | a **RAFT** fine-tune on golden+distractor examples (LoRA-sized) |

```python
# real embeddings + retrieval, dropped straight into retrieve():
from sentence_transformers import SentenceTransformer
enc = SentenceTransformer('all-MiniLM-L6-v2')
CHUNK_VECS = {cid: enc.encode(t) for cid, t in CORPUS.items()}
```

Frameworks that give you this out of the box: **LlamaIndex** (retrieval), **RAGAS** (the eval harness), **gorilla/raft** (the RAFT recipe), **PEFT/Unsloth** (the LoRA fine-tune).

## Key takeaways

- You built a support bot as a chain of **verifiable milestones**: skeleton → retrieval → grounded generation → evaluation → RAFT robustness → integration.
- **Grounding + citation** is what separates a support bot from a chatbot that makes things up.
- **Evaluate retrieval and generation separately** — a wrong answer is usually a retrieval miss.
- **RAFT** keeps the bot useful when retrieval degrades by falling back on memorized knowledge.

Back to the [Support Chatbot project](https://ml-viz-ruby.vercel.app/projects/support-chatbot) · next project: [Ask-the-Web Agent](https://ml-viz-ruby.vercel.app/projects/ask-the-web-agent).